# Runtime Reliability Simulation

Discrete-time EDF simulation with primary/backup placement, checkpoint overhead,
rollback, transient faults, disruptive faults, and backup activation. The experiment
runner is disabled by default and writes generated reports under `artifacts/`.


### Fault-rate time units

`alpha_0 = 1e-6` is expressed in **faults per microsecond**. Simulator time is measured in **milliseconds**, so `fault_rate_formula_3(...)` converts the rate to faults per millisecond by multiplying it by `1000`.

In [ ]:

# ============================================================
# Imports + basic helpers
# ============================================================
import gym
from gym import spaces
import pandas as pd
import numpy as np
import random
import math
from collections import deque
from decimal import Decimal, ROUND_HALF_UP
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import os
from itertools import zip_longest
from scipy.stats import truncnorm
import csv
from functools import reduce
import torch
import torch.nn as nn
import copy

# Task/job times are milliseconds in this project.
TIME_UNIT = "ms"

# Fault-rate model parameters.
ALPHA_0 = 1e-6
D_FAULT = 2


def lcm(a, b):
    return abs(int(a) * int(b)) // math.gcd(int(a), int(b))


def lcm_of_list(numbers):
    numbers = [int(x) for x in numbers]
    return reduce(lcm, numbers)


def safe_remove(lst, item):
    if item in lst:
        lst.remove(item)
        return True
    return False


In [ ]:

# ============================================================
# Reliability + checkpoint formulas from the reliability formulation
# ============================================================


def fault_rate_formula_3(f, f_min, f_max, alpha_0=ALPHA_0, d=D_FAULT):
    """
    Formula 3 fault-rate model.

    Important unit convention:
        alpha_0 = 10^-6 is interpreted as faults per microsecond.
        The simulator executes jobs in milliseconds.
        Therefore the fault rate is converted to faults/ms before it is used.

    Normalized DVFS fault-rate form:
        lambda_us(f) = alpha_0 * 10^( d * (1 - normalized_frequency) )

    Therefore:
        f = f_max => lambda_us = alpha_0
        f = f_min => lambda_us = alpha_0 * 10^d

    Returned value:
        lambda_ms(f) = lambda_us(f) * 1000
    """
    if f_max == f_min:
        normalized_frequency = 1.0
    else:
        normalized_frequency = (f - f_min) / (f_max - f_min)

    normalized_frequency = max(0.0, min(1.0, normalized_frequency))

    lambda_per_microsecond = alpha_0 * (10 ** (d * (1.0 - normalized_frequency)))
    lambda_per_millisecond = lambda_per_microsecond * 1000.0

    return lambda_per_millisecond


def poisson_event(rate_per_ms, dt_ms=1.0):
    """Return True if an event occurs in this time step."""
    p = 1.0 - math.exp(-rate_per_ms * dt_ms)
    return random.random() < p


def reliability_with_checkpoint(exec_time_ms, f, f_min, f_max, k, alpha_0=ALPHA_0, d=D_FAULT):
    """
    Checkpoint reliability (Formula 16-4):
        R_RP/CP(T_i) = sum_{s=0}^{k_i} ((lambda(f_i) W(vf_i))^s e^{-lambda(f_i) W(vf_i)}) / s!

    The Poisson probability of tolerating up to k faults includes s=0, so the
    summation is evaluated over s=0..k.
    """
    lam = fault_rate_formula_3(f, f_min, f_max, alpha_0, d)
    x = lam * exec_time_ms
    return sum(((x ** s) * math.exp(-x)) / math.factorial(s) for s in range(0, int(k) + 1))


def reliability_no_checkpoint(exec_time_ms, f, f_min, f_max, alpha_0=ALPHA_0, d=D_FAULT):
    """Backup reliability without checkpoint: R_B(T_i)=exp(-lambda(f_i)*W_i)."""
    lam = fault_rate_formula_3(f, f_min, f_max, alpha_0, d)
    return math.exp(-lam * exec_time_ms)


def total_task_reliability(R_cp, R_backup):
    """
    Total task reliability (Formula 17-4):
        R_total(T_i) = R_RP/CP(T_i) + (1 - R_RP/CP(T_i)) R_B(T_i)
    """
    return R_cp + (1.0 - R_cp) * R_backup


def system_reliability(tasks):
    """
    System reliability (Formula 18-4):
        R_system = product_i R_total(T_i)
    """
    r = 1.0
    for task in tasks:
        r *= getattr(task, "total_reliability", 1.0)
    return r


def checkpointed_wcet_formula_23(t_i, k_i, n_i, c_i, r_i):
    """
    Checkpoint-adjusted worst-case execution time (Formula 23-4):
        WC_i(t_i, k_i, n_i) = t_i + n_i c_i + k_i (t_i/n_i + r_i)

    t_i : execution time without checkpoint, in ms
    k_i : number of transient faults to tolerate
    n_i : number of checkpoints
    c_i : checkpoint save overhead, in ms
    r_i : recovery overhead, in ms
    """
    if k_i <= 0 or n_i <= 0:
        return int(math.ceil(t_i))
    return int(math.ceil(t_i + n_i * c_i + k_i * ((t_i / n_i) + r_i)))


def checkpoint_count_formula_24(t_i, k_i, c_i, r_i):
    """
    Checkpoint count selection (Formula 24-4).

    Candidate checkpoint counts:
        n^- = floor( sqrt( k_i * wc_i / (O_i + O_f) ) )
        n^+ = ceil(  sqrt( k_i * wc_i / (O_i + O_f) ) )

    and chooses n^- or n^+ using the threshold:
        wc_i <= n^- (n^- + 1) (O_i + O_f) / k_i  => n^-
        otherwise                                  => n^+

    In this code:
        wc_i = t_i
        O_i = c_i  checkpoint overhead
        O_f = r_i  recovery overhead

    The floor/ceil candidates are selected using the threshold above.
    """
    if k_i <= 0:
        return 0
    overhead_sum = c_i + r_i
    if overhead_sum <= 0:
        return 1

    root = math.sqrt((k_i * t_i) / overhead_sum)
    n_minus = max(1, int(math.floor(root)))
    n_plus = max(1, int(math.ceil(root)))

    threshold = (n_minus * (n_minus + 1) * overhead_sum) / k_i
    if t_i <= threshold:
        return n_minus
    return n_plus


def checkpoint_times_by_formula(t_i, n_i):
    """Uniform checkpoint placement implied by t_i/n_i in Formula 23-4."""
    if n_i <= 0:
        return []
    return [int(math.ceil(j * t_i / n_i)) for j in range(1, int(n_i) + 1)]


In [ ]:

# ============================================================
# Shared state
# ============================================================
class Shared:
    def __init__(self):
        self.slist = []
        self.is_primary_set = False
        self.done = False
        self.success = False
        self.failed = False
        self.final_job = None
        self.final_time = -1
        self.primary_failed = False
        self.backup_activated = False


In [ ]:

# ============================================================
# Task
# ============================================================
class Task:
    def __init__(self, name, vf, initial_exec_task_time, deadline, alpha_c, I0_exp_term,
                 peak_aC, min_aC, shared, WCET, worst_vf, exec_set, vf_levels, worst_lvl):
        self.name = name
        self.init_v, self.init_f = vf
        self.initial_exec_task_time = int(math.ceil(initial_exec_task_time))
        self.current_exec_time = int(math.ceil(initial_exec_task_time))
        self.alpha_C = alpha_c
        self.I0_exp_term = I0_exp_term
        self.peak_aC = peak_aC
        self.min_aC = min_aC
        self.period = int(math.ceil(deadline))
        self.deadline = int(math.ceil(deadline))
        self.shared_tasks = shared
        self.arrival = 0
        self.fault = False
        self.WCET = int(math.ceil(WCET))
        self.worst_v, self.worst_f = worst_vf
        self.exec_set = {int(k): int(math.ceil(v)) for k, v in exec_set.items()}
        self.vf_levels = vf_levels
        self.v, self.f = vf
        self.core = None
        self.is_primary = False
        self.job_id = 0
        self.slack = 0
        self.current_job_id = -1
        self.worst_lvl = worst_lvl

        # Checkpoint parameters.
        self.k_faults = 1
        self.checkpoint_overhead = 1
        self.recovery_overhead = 1
        self.n_checkpoints = 0
        self.checkpoint_times = []
        self.checkpoint_interval = 0
        self.checkpoint_count_by_lvl = {}
        self.checkpoint_times_by_lvl = {}
        self.exec_set_cp = {}
        self.WCET_cp = self.WCET

        # Reliability values.
        self.R_cp = 0.0
        self.R_backup = 0.0
        self.total_reliability = 0.0

    def get_util(self):
        return self.current_exec_time / self.period

    def get_vf_level(self, v, f):
        for lvl, vf in self.vf_levels.items():
            if vf["voltage"] == v and vf["frequency"] == f:
                return lvl
        return None

    def update(self, v, f, core, checkpointed=False):
        self.f = f
        self.v = v
        self.core = core
        lvl = self.get_vf_level(v, f)
        if checkpointed and lvl is not None and lvl in self.exec_set_cp:
            self.current_exec_time = self.exec_set_cp[lvl]
        elif lvl is not None:
            self.current_exec_time = self.exec_set[lvl]
        else:
            self.current_exec_time = int(math.ceil(self.initial_exec_task_time / (f / self.init_f)))

    def configure_checkpointing(self, k_i=1, c_i=1, r_i=1):
        self.k_faults = int(k_i)
        self.checkpoint_overhead = int(math.ceil(c_i))
        self.recovery_overhead = int(math.ceil(r_i))
        self.checkpoint_count_by_lvl = {}
        self.checkpoint_times_by_lvl = {}
        self.exec_set_cp = {}

        for lvl, t_i in self.exec_set.items():
            n_i = checkpoint_count_formula_24(
                t_i=t_i,
                k_i=self.k_faults,
                c_i=self.checkpoint_overhead,
                r_i=self.recovery_overhead
            )
            self.checkpoint_count_by_lvl[lvl] = n_i
            self.checkpoint_times_by_lvl[lvl] = checkpoint_times_by_formula(t_i, n_i)
            self.exec_set_cp[lvl] = checkpointed_wcet_formula_23(
                t_i=t_i,
                k_i=self.k_faults,
                n_i=n_i,
                c_i=self.checkpoint_overhead,
                r_i=self.recovery_overhead
            )

        self.n_checkpoints = checkpoint_count_formula_24(
            t_i=self.WCET,
            k_i=self.k_faults,
            c_i=self.checkpoint_overhead,
            r_i=self.recovery_overhead
        )
        self.checkpoint_times = checkpoint_times_by_formula(self.WCET, self.n_checkpoints)
        self.checkpoint_interval = self.WCET / self.n_checkpoints if self.n_checkpoints > 0 else 0
        self.WCET_cp = checkpointed_wcet_formula_23(
            t_i=self.WCET,
            k_i=self.k_faults,
            n_i=self.n_checkpoints,
            c_i=self.checkpoint_overhead,
            r_i=self.recovery_overhead
        )

    def update_reliability(self, env):
        f_min = min(vf["frequency"] for vf in env.vf_levels.values())
        f_max = max(vf["frequency"] for vf in env.vf_levels.values())
        self.R_cp = reliability_with_checkpoint(
            exec_time_ms=self.current_exec_time,
            f=self.f,
            f_min=f_min,
            f_max=f_max,
            k=self.k_faults,
            alpha_0=env.lambda_0,
            d=env.d_fault
        )
        self.R_backup = reliability_no_checkpoint(
            exec_time_ms=self.WCET,
            f=self.worst_f,
            f_min=f_min,
            f_max=f_max,
            alpha_0=env.lambda_0,
            d=env.d_fault
        )
        self.total_reliability = total_task_reliability(self.R_cp, self.R_backup)
        return self.total_reliability

    def get_peak_p(self):
        return self.peak_aC * (self.v ** 2) * self.f + self.I0_exp_term * self.v

    def get_min_p(self):
        return self.min_aC * (self.v ** 2) * self.f + self.I0_exp_term * self.v

    def get_avg_p(self):
        return self.alpha_C * (self.v ** 2) * self.f + self.I0_exp_term * self.v

    def get_vf_lvl_peak(self, i):
        return self.peak_aC * (self.vf_levels[i]["voltage"] ** 2) * self.vf_levels[i]["frequency"] + self.I0_exp_term * self.vf_levels[i]["voltage"]


In [ ]:

# ============================================================
# Job with detailed checkpoint runtime state
# ============================================================
class Job(Task):
    def __init__(self, task: Task, shared):
        super().__init__(
            name=task.name,
            vf=(task.init_v, task.init_f),
            initial_exec_task_time=task.initial_exec_task_time,
            deadline=task.deadline,
            alpha_c=task.alpha_C,
            I0_exp_term=task.I0_exp_term,
            peak_aC=task.peak_aC,
            min_aC=task.min_aC,
            shared=shared,
            WCET=task.WCET,
            worst_vf=(task.worst_v, task.worst_f),
            exec_set=task.exec_set,
            vf_levels=task.vf_levels,
            worst_lvl=task.worst_lvl
        )
        self.task = task
        self.min_start_time = -1
        self.vf_is_set = False
        self.real_exec = 0
        self.is_real_exec_set = False
        self.start_time = -1
        self.finish_time = -1
        self.current_peak = -1
        self.reliability = -1
        self.real_reliability = -1
        self.status = "created"

        # Runtime checkpoint/fault state.
        self.runtime_initialized = False
        self.useful_work_ms = 0
        self.useful_done_ms = 0
        self.last_checkpoint_useful_ms = 0
        self.next_checkpoint_index = 0
        self.active_checkpoint_times = []
        self.checkpoint_save_left = 0
        self.recovery_left = 0
        self.completed = False
        self.failed = False
        self.fault = False
        self.transient_fault_events = 0
        self.disruptive_fault_events = 0
        self.rollback_events = 0
        self.reexecuted_work_ms = 0
        self.total_runtime_ms = 0
        self.failure_reason = ""

    def job_update(self, task: Task):
        self.task = task
        self.arrival = task.arrival
        self.fault = False
        self.v, self.f = task.v, task.f
        self.v_policy, self.f_policy = task.v, task.f
        self.core = task.core
        self.is_primary = task.is_primary
        self.current_exec_time = task.current_exec_time
        self.job_id = task.job_id

        self.k_faults = task.k_faults
        self.checkpoint_overhead = task.checkpoint_overhead
        self.recovery_overhead = task.recovery_overhead
        self.n_checkpoints = task.n_checkpoints
        self.checkpoint_times = list(task.checkpoint_times)
        self.checkpoint_interval = task.checkpoint_interval
        self.checkpoint_count_by_lvl = dict(task.checkpoint_count_by_lvl)
        self.checkpoint_times_by_lvl = dict(task.checkpoint_times_by_lvl)
        self.exec_set_cp = dict(task.exec_set_cp)
        self.WCET_cp = task.WCET_cp

    def apply_DVFS(self, v, f):
        self.task.f = f
        self.task.v = v
        self.f = f
        self.v = v
        lvl = self.task.get_vf_level(v, f)

        if self.is_primary and lvl is not None:
            self.current_exec_time = self.task.exec_set_cp[lvl]
            self.useful_work_ms = self.task.exec_set[lvl]
            self.n_checkpoints = self.task.checkpoint_count_by_lvl[lvl]
            self.active_checkpoint_times = list(self.task.checkpoint_times_by_lvl[lvl])
        elif lvl is not None:
            self.current_exec_time = self.task.exec_set[lvl]
            self.useful_work_ms = self.task.exec_set[lvl]
            self.n_checkpoints = 0
            self.active_checkpoint_times = []
        else:
            base = int(math.ceil(self.task.initial_exec_task_time / (f / self.task.init_f)))
            self.useful_work_ms = base
            if self.is_primary:
                self.n_checkpoints = checkpoint_count_formula_24(base, self.k_faults, self.checkpoint_overhead, self.recovery_overhead)
                self.active_checkpoint_times = checkpoint_times_by_formula(base, self.n_checkpoints)
                self.current_exec_time = checkpointed_wcet_formula_23(base, self.k_faults, self.n_checkpoints, self.checkpoint_overhead, self.recovery_overhead)
            else:
                self.n_checkpoints = 0
                self.active_checkpoint_times = []
                self.current_exec_time = base

        self.task.current_exec_time = self.current_exec_time
        self.real_exec = self.current_exec_time
        self.is_real_exec_set = True
        self.runtime_initialized = False

    def initialize_runtime(self):
        if self.runtime_initialized:
            return
        if self.useful_work_ms <= 0:
            lvl = self.task.get_vf_level(self.v, self.f)
            self.useful_work_ms = self.task.exec_set.get(lvl, self.current_exec_time)
        self.useful_done_ms = 0
        self.last_checkpoint_useful_ms = 0
        self.next_checkpoint_index = 0
        self.checkpoint_save_left = 0
        self.recovery_left = 0
        self.completed = False
        self.failed = False
        self.status = "running"
        self.runtime_initialized = True

    def get_current_util(self):
        return self.current_exec_time / self.period

    def get_current_peak(self):
        return self.task.peak_aC * (self.task.v ** 2) * self.task.f + self.task.I0_exp_term * self.task.v

    def get_rep_num(self):
        return len(self.shared_tasks.slist) - 1

    def is_done_or_failed(self):
        return self.completed or self.failed


In [ ]:

# ============================================================
# Core
# ============================================================
class Core:
    def __init__(self, id):
        self.id = id
        self.ready = []
        self.running_replica = []
        self.mapped_for_slack = []
        self.primary = []
        self.replica_st = []
        self.replica_dl = []
        self.current_utilization = 0
        self.utilization = 0
        self.slack_time = 0
        self.running = None
        self.disrupted_until = -1

    def EDF(self):
        self.primary.sort(key=lambda t: t.deadline)
        self.replica_st.sort(key=lambda t: t.min_start_time)
        self.replica_dl.sort(key=lambda t: t.deadline)

    def remove_replica(self, j):
        existed_running = self.running == j
        safe_remove(self.replica_dl, j)
        safe_remove(self.replica_st, j)
        safe_remove(self.ready, j)
        if self.running == j:
            self.running = None
        return existed_running

    def remove_primary(self, j):
        safe_remove(self.primary, j)
        safe_remove(self.ready, j)
        if self.running == j:
            self.running = None


In [ ]:

# ============================================================
# QoS
# ============================================================
def utility(fi, Di, x=2):
    fi = np.asarray(fi, dtype=float)
    Di = np.asarray(Di, dtype=float)
    if x <= 1:
        raise ValueError("x must be > 1")
    UF = np.zeros_like(fi, dtype=float)
    on_time = fi <= Di
    soft = (fi > Di) & (fi <= x * Di)
    UF[on_time] = 1.0
    UF[soft] = (Di[soft] - fi[soft]) / (Di[soft] * (x - 1)) + 1.0
    return UF


def qos_system_no_backup(P, f_main, D, x=2):
    P = np.asarray(P, dtype=float)
    f_main = np.asarray(f_main, dtype=float)
    D = np.asarray(D, dtype=float)
    if len(f_main) == 0:
        return 0.0, []
    if not (len(P) == len(f_main) == len(D)):
        raise ValueError("P, f_main, and D must have the same length")
    UF = utility(f_main, D, x)
    qos = np.sum(P * UF) / len(P)
    return float(qos), UF


In [ ]:

# ============================================================
# Environment
# ============================================================
class Envir(gym.Env):
    def __init__(self, initial_voltage, initial_frequency, cores, main_tasks, alpha):
        super().__init__()
        self.cores = cores
        self.main_tasks = main_tasks
        self.current_task_count = 0
        self.static = 0
        self.alpha = alpha
        self.data = []
        self.train_data = []

        # VF levels: frequency in Hz, voltage in V
        self.vf_levels = {
            0: {"frequency": 800e6, "voltage": 0.9},
            1: {"frequency": 1.1e9, "voltage": 0.95},
            2: {"frequency": 1.4e9, "voltage": 1.0},
            3: {"frequency": 1.7e9, "voltage": 1.05},
            4: {"frequency": 2.0e9, "voltage": 1.1},
        }
        self.initial_voltage = self.vf_levels[len(self.vf_levels) - 1]["voltage"]
        self.initial_frequency = self.vf_levels[len(self.vf_levels) - 1]["frequency"]

        # Fault model from Formula 3.
        self.lambda_0 = ALPHA_0
        self.d_fault = D_FAULT

        # The total Formula-3 fault rate is split into transient and disruptive faults.
        # Both still come from Formula 3; these ratios only classify the type.
        self.transient_fault_ratio = 0.8
        self.disruptive_fault_ratio = 0.2

        # Default checkpoint parameters, in ms.
        self.default_k_faults = 2
        self.default_checkpoint_overhead = 1
        self.default_recovery_overhead = 1

        # Optional downtime after disruptive core fault, in ms.
        self.core_reboot_time = 0

        self.system_tasks = []
        self.state = None
        self.action = None
        self.total_util = 0
        self.reset()

    def reset(self):
        self.state = {core: 0.0 for core in range(self.cores)}
        self.action = None
        self.total_util = 0
        self.train_data = self.data.copy()
        self.current_task_count = 0
        self.system_tasks.clear()
        return self.state

    def read_data(self, input_files):
        try:
            for file in input_files:
                task_name = os.path.basename(file).replace(".csv", "")
                task_data = pd.read_csv(file)

                initial_exec_task_time = int(math.ceil(task_data["#"].iloc[0]))
                self.static = task_data["static"].iloc[0]
                alpha_C = task_data["alpha_C"].iloc[0]
                I0_exp_term = task_data["I0_exp_term"].iloc[0]
                peak = task_data["peak"].iloc[0]
                deadline = int(math.ceil(task_data["deadline"].iloc[0]))

                exec_set = {i: 0 for i in range(len(self.vf_levels))}
                peak_list = []
                worst_lvl = -1
                tsp_limit = globals().get("TSP", float("inf"))

                for i in range(len(self.vf_levels)):
                    exec_set[i] = int(math.ceil(initial_exec_task_time / (self.vf_levels[i]["frequency"] / self.initial_frequency)))
                    peak_power_i = peak * (self.vf_levels[i]["voltage"] ** 2) * self.vf_levels[i]["frequency"] + I0_exp_term * self.vf_levels[i]["voltage"]
                    peak_list.append((peak_power_i, i))

                peak_list.sort(key=lambda t: t[0], reverse=True)
                for tup in peak_list:
                    if tup[0] <= tsp_limit:
                        worst_lvl = tup[1]
                        break
                if worst_lvl == -1:
                    worst_lvl = 0

                WCET = int(math.ceil(initial_exec_task_time / (self.vf_levels[worst_lvl]["frequency"] / self.initial_frequency)))

                task = Task(
                    task_name,
                    (self.initial_voltage, self.initial_frequency),
                    initial_exec_task_time,
                    deadline,
                    alpha_C,
                    I0_exp_term,
                    peak,
                    task_data["min"].head(1).mean(),
                    Shared(),
                    WCET,
                    (self.vf_levels[worst_lvl]["voltage"], self.vf_levels[worst_lvl]["frequency"]),
                    exec_set,
                    self.vf_levels,
                    worst_lvl
                )

                task.configure_checkpointing(
                    k_i=self.default_k_faults,
                    c_i=self.default_checkpoint_overhead,
                    r_i=self.default_recovery_overhead
                )
                self.data.append(task)
        except Exception as e:
            print(f"An error occurred: {e}")


In [ ]:

# ============================================================
# Slack + mapping + job creation
# ============================================================
def calculate_slack(cores, hyper_period):
    for c in cores:
        total_WCET = 0
        for job in c.mapped_for_slack:
            total_WCET += job.current_exec_time
        c.slack_time = hyper_period - total_WCET


def define_slack(cores, data, hyper_period):
    for c in cores:
        w_t = []
        for d in data:
            if c != d.core:
                continue
            w_t.append((hyper_period / d.period) * d.WCET_cp)
        w_t = sum(w_t)
        if w_t == 0:
            continue
        for d in data:
            if c != d.core:
                continue
            d.slack = int((((hyper_period / d.period) * d.WCET_cp) / w_t) * c.slack_time)


def map(data, ustate, env, cores):
    """
    Primary uses checkpointed WCET.
    Exactly one backup is mapped to a different core and uses normal WCET.
    """
    # Primary mapping.
    for task in data:
        min_index = min(ustate, key=ustate.get)
        core = cores[min_index]
        task.is_primary = True
        task.update(
            env.vf_levels[len(env.vf_levels) - 1]["voltage"],
            env.vf_levels[len(env.vf_levels) - 1]["frequency"],
            core,
            checkpointed=True
        )
        ustate[min_index] += task.WCET_cp / task.period
        task.shared_tasks.slist.append(task)
        task.core.current_utilization += task.WCET_cp / task.period

    # One backup per task on a different core.
    backup_data = sorted(data, key=lambda d: d.deadline)
    for task in backup_data:
        candidate_utils = ustate.copy()
        candidate_utils[task.core.id] = float("inf")
        backup_core_id = min(candidate_utils, key=candidate_utils.get)
        backup_core = cores[backup_core_id]

        backup_task = copy.copy(task)
        backup_task.shared_tasks = task.shared_tasks
        backup_task.is_primary = False
        backup_task.core = backup_core
        backup_task.update(task.worst_v, task.worst_f, backup_core, checkpointed=False)
        backup_task.current_exec_time = backup_task.WCET

        backup_util = backup_task.WCET / backup_task.period
        if ustate[backup_core_id] + backup_util > 1.0:
            print(f"Warning: backup of {task.name} maps to core {backup_core_id}, but utilization exceeds 1.0")
        ustate[backup_core_id] += backup_util
        backup_task.shared_tasks.slist.append(backup_task)


def create_jobs(data, hyper_period, cores, env):
    jobs = []
    arrivals = []
    for t in range(int(hyper_period)):
        for task in data:
            if task.arrival == t:
                group = Shared()

                p_job = Job(task, group)
                p_job.job_update(task)
                p_job.is_primary = True
                p_job.core = task.core
                p_job.current_exec_time = task.WCET_cp
                p_job.job_id = task.job_id
                p_job.arrival = task.arrival
                p_job.deadline = task.deadline
                p_job.shared_tasks = group
                group.slist.append(p_job)
                jobs.append(p_job)
                arrivals.append(task.arrival)
                p_job.core.mapped_for_slack.append(p_job)

                for rep_template in task.shared_tasks.slist:
                    if rep_template is task or rep_template.is_primary:
                        continue
                    r_job = Job(rep_template, group)
                    r_job.job_update(rep_template)
                    r_job.is_primary = False
                    r_job.core = rep_template.core
                    r_job.current_exec_time = rep_template.WCET
                    r_job.useful_work_ms = rep_template.WCET
                    r_job.job_id = task.job_id
                    r_job.arrival = task.arrival
                    r_job.deadline = task.deadline
                    r_job.min_start_time = r_job.deadline - r_job.WCET
                    r_job.shared_tasks = group
                    group.slist.append(r_job)
                    jobs.append(r_job)

                task.arrival += task.period
                task.deadline += task.period
                task.job_id += 1
    arrivals = sorted(list(set(arrivals)))
    return jobs, arrivals


def bounded_normal(min_val, max_val, center, spread_factor=0.25):
    if max_val <= min_val:
        return center
    range_val = max_val - min_val
    std_dev = spread_factor * range_val
    if std_dev <= 0:
        return center
    a, b = (min_val - center) / std_dev, (max_val - center) / std_dev
    return truncnorm.rvs(a, b, loc=center, scale=std_dev)


In [ ]:

# ============================================================
# Arrival and dispatch
# ============================================================
def manage_arrival(arrival_queue, cores, job_queue, time):
    new_arrived = False
    arr = -1
    if arrival_queue:
        arr = arrival_queue.popleft()
    if time == arr:
        while True:
            if not job_queue:
                break
            job = job_queue.popleft()
            if job.arrival != arr:
                job_queue.appendleft(job)
                break
            if not job.is_primary:
                job.core.replica_st.append(job)
                job.core.replica_dl.append(job)
                job.core.ready.append(job)
            else:
                job.core.primary.append(job)
                job.core.ready.append(job)
            new_arrived = True
    else:
        if arr != -1:
            arrival_queue.appendleft(arr)
    return new_arrived


def logic_of_replicas(c, time):
    available = [j for j in c.replica_dl if j.min_start_time <= time and not j.shared_tasks.done]
    if not available:
        return
    available.sort(key=lambda j: (j.deadline, j.min_start_time))
    c.running = available[0]


def dispatch(cores, time):
    for c in cores:
        if time < c.disrupted_until:
            continue
        if (not c.primary) and (not c.replica_dl):
            continue

        c.EDF()
        selected = c.running

        available_primary = [j for j in c.primary if not j.shared_tasks.done]
        available_replica = [j for j in c.replica_dl if j.min_start_time <= time and not j.shared_tasks.done]

        candidates = []
        if available_primary:
            candidates.append(min(available_primary, key=lambda j: j.deadline))
        if available_replica:
            candidates.append(min(available_replica, key=lambda j: j.deadline))

        if not candidates:
            continue

        best = min(candidates, key=lambda j: j.deadline)
        if selected is None or best.deadline < selected.deadline:
            c.running = best


In [ ]:

# ============================================================
# DVFS
# ============================================================
def DVFS(cores, time, vf_levels):
    for c in cores:
        if not c.running or c.running.vf_is_set:
            continue

        job = c.running
        worst_lvl = job.worst_lvl

        if not job.is_primary:
            old_util = job.get_current_util()
            job.apply_DVFS(job.worst_v, job.worst_f)
            new_util = job.get_current_util()
            c.current_utilization += (new_util - old_util)
            job.task.current_job_id = job.job_id
            if job.task not in c.running_replica:
                c.running_replica.append(job.task)
        else:
            old_util = job.get_current_util()
            selected_lvl = None
            selected_util = -1

            # Select the slowest feasible level that keeps core utilization <= 1.0.
            for lvl in range(worst_lvl + 1):
                if lvl not in job.task.exec_set_cp:
                    continue
                candidate_exec = job.task.exec_set_cp[lvl]
                candidate_util = c.current_utilization + (candidate_exec / job.task.period - old_util)
                if 0 <= candidate_util <= 1.0 and candidate_util > selected_util:
                    selected_util = candidate_util
                    selected_lvl = lvl

            if selected_lvl is None:
                selected_lvl = worst_lvl

            job.apply_DVFS(vf_levels[selected_lvl]["voltage"], vf_levels[selected_lvl]["frequency"])
            new_util = job.get_current_util()
            c.current_utilization += (new_util - old_util)
            job.task.update_reliability(globals().get("env", None)) if "env" in globals() else None

        job.vf_is_set = True


In [ ]:

# ============================================================
# Runtime fault injection + checkpoint execution
# ============================================================
def get_fault_rates_for_job(job, env):
    f_min = min(vf["frequency"] for vf in env.vf_levels.values())
    f_max = max(vf["frequency"] for vf in env.vf_levels.values())
    total_rate = fault_rate_formula_3(job.f, f_min, f_max, env.lambda_0, env.d_fault)
    transient_rate = total_rate * env.transient_fault_ratio
    disruptive_rate = total_rate * env.disruptive_fault_ratio
    return transient_rate, disruptive_rate, total_rate


def activate_backup_for_failed_primary(primary_job, time):
    group = primary_job.shared_tasks
    group.primary_failed = True
    group.backup_activated = True
    for rep in group.slist:
        if rep.is_primary:
            continue
        if rep.completed or rep.failed:
            continue
        rep.min_start_time = time
        rep.status = "backup_activated"
        if rep.core and rep not in rep.core.ready:
            rep.core.ready.append(rep)
        if rep.core and rep not in rep.core.replica_st:
            rep.core.replica_st.append(rep)
        if rep.core and rep not in rep.core.replica_dl:
            rep.core.replica_dl.append(rep)
        rep.core.EDF()


def fail_job(job, time, reason, disruptive=False):
    job.failed = True
    job.fault = True
    job.finish_time = time
    job.status = "failed"
    job.failure_reason = reason
    if disruptive:
        job.disruptive_fault_events += 1
    if job.is_primary:
        activate_backup_for_failed_primary(job, time)
    else:
        # Backup failed. If primary already failed and no other backup exists, group fails.
        group = job.shared_tasks
        alive_backup = any((not j.is_primary) and (not j.failed) and (not j.completed) for j in group.slist)
        primary_alive = any(j.is_primary and (not j.failed) and (not j.completed) for j in group.slist)
        if not alive_backup and not primary_alive and not group.done:
            group.done = True
            group.success = False
            group.failed = True
            group.final_job = job
            group.final_time = time


def complete_job_success(job, time):
    job.completed = True
    job.finish_time = time
    job.status = "success"
    group = job.shared_tasks
    if not group.done:
        group.done = True
        group.success = True
        group.failed = False
        group.final_job = job
        group.final_time = time


def execute_one_ms(job, env, time):
    """
    Execute one millisecond of the job and inject faults during execution.

    Transient fault:
        - primary rolls back to latest checkpoint, pays recovery overhead, then re-executes.
        - if transient count exceeds k_i, primary fails and backup is activated.
        - backup has no checkpoint, so transient fault fails the backup.

    Disruptive fault:
        - checkpoint cannot recover it.
        - primary fails and backup is activated.
        - backup fails if it suffers this fault.
    """
    job.initialize_runtime()
    if job.completed or job.failed:
        return

    transient_rate, disruptive_rate, _ = get_fault_rates_for_job(job, env)

    # Faults are injected at each simulation time step, not at the end.
    if poisson_event(disruptive_rate, dt_ms=1.0):
        fail_job(job, time, "disruptive_core_fault", disruptive=True)
        return

    if poisson_event(transient_rate, dt_ms=1.0):
        job.transient_fault_events += 1
        if job.is_primary:
            if job.transient_fault_events > job.k_faults:
                fail_job(job, time, "too_many_transient_faults", disruptive=False)
                return
            lost_work = job.useful_done_ms - job.last_checkpoint_useful_ms
            job.reexecuted_work_ms += max(0, lost_work)
            job.useful_done_ms = job.last_checkpoint_useful_ms
            job.recovery_left += job.recovery_overhead
            job.rollback_events += 1
            job.status = "rollback_to_checkpoint"
            return
        else:
            fail_job(job, time, "transient_fault_on_backup_without_checkpoint", disruptive=False)
            return

    # Recovery overhead after rollback.
    if job.recovery_left > 0:
        job.recovery_left -= 1
        job.total_runtime_ms += 1
        job.real_exec = max(0, job.real_exec - 1)
        return

    # Checkpoint saving overhead.
    if job.checkpoint_save_left > 0:
        job.checkpoint_save_left -= 1
        job.total_runtime_ms += 1
        job.real_exec = max(0, job.real_exec - 1)
        return

    # Useful execution.
    job.useful_done_ms += 1
    job.total_runtime_ms += 1
    job.real_exec = max(0, job.real_exec - 1)

    # Save checkpoint when reaching the next checkpoint position.
    if job.is_primary and job.next_checkpoint_index < len(job.active_checkpoint_times):
        cp_pos = job.active_checkpoint_times[job.next_checkpoint_index]
        if job.useful_done_ms >= cp_pos:
            job.last_checkpoint_useful_ms = min(job.useful_done_ms, job.useful_work_ms)
            job.next_checkpoint_index += 1
            job.checkpoint_save_left += job.checkpoint_overhead
            job.status = "saving_checkpoint"

    # Finish only after useful work is done and no checkpoint/recovery overhead remains.
    if job.useful_done_ms >= job.useful_work_ms and job.checkpoint_save_left == 0 and job.recovery_left == 0:
        complete_job_success(job, time)


def execute(cores, time, power_trace, util_trace, total_jobs, total_finished_jobs, faulty_jobs_counter, hyper_period, env, fault_report):
    u_tup = [0 for _ in cores]
    for c in cores:
        if time < c.disrupted_until:
            power_trace[c.id].append(0)
            continue
        if not c.running:
            power_trace[c.id].append(0)
            continue

        job = c.running
        if job.start_time == -1:
            job.start_time = time

        prev_transient = job.transient_fault_events
        prev_disruptive = job.disruptive_fault_events
        prev_failed = job.failed
        prev_completed = job.completed

        execute_one_ms(job, env, time)

        fault_report["transient_fault_events"] += job.transient_fault_events - prev_transient
        fault_report["disruptive_fault_events"] += job.disruptive_fault_events - prev_disruptive
        if (not prev_failed) and job.failed:
            fault_report["faulty_jobs"] += 1
            if job.is_primary:
                fault_report["faulty_primary_jobs"] += 1
            else:
                fault_report["faulty_backup_jobs"] += 1
        if (not prev_completed) and job.completed:
            fault_report["successful_jobs"] += 1

        if random.random() < 0.1:
            power_trace[c.id].append(job.get_peak_p())
        else:
            power_trace[c.id].append(bounded_normal(job.get_min_p(), job.get_peak_p(), job.get_avg_p()))
        u_tup[c.id] = c.current_utilization

    util_trace.append(tuple(u_tup))


In [ ]:

# ============================================================
# Finish handling and cancellation
# ============================================================
def cancel_other_copies(success_job):
    for rep in list(success_job.shared_tasks.slist):
        if rep is success_job:
            continue
        if rep.core is None:
            continue
        if rep.is_primary:
            rep.core.remove_primary(rep)
        else:
            existed = rep.core.remove_replica(rep)
            if (rep.task in rep.core.running_replica) and (rep.job_id > rep.task.current_job_id):
                safe_remove(rep.core.running_replica, rep.task)
                rep.core.current_utilization -= rep.get_current_util()
        rep.status = "cancelled"
        rep.finish_time = success_job.finish_time


def remove_finished_or_failed_from_core(core, job):
    if job.is_primary:
        core.remove_primary(job)
    else:
        core.remove_replica(job)
        if (job.task in core.running_replica) and (job.job_id > job.task.current_job_id):
            safe_remove(core.running_replica, job.task)
            core.current_utilization -= job.get_current_util()
    if core.running is job:
        core.running = None


def handle_finished_jobs(cores, time, TSP):
    TSP_violate_counter = 0
    total_finished_jobs = 0
    deadline_missed_counter = 0
    faulty_jobs_counter = 0

    for c in cores:
        job = c.running
        if not job:
            continue
        if not job.completed and not job.failed:
            continue

        if job.completed:
            if job.get_peak_p() > TSP:
                TSP_violate_counter += 1
            if job.finish_time > job.deadline:
                deadline_missed_counter += 1
            total_finished_jobs += 1
            cancel_other_copies(job)
            remove_finished_or_failed_from_core(c, job)

        elif job.failed:
            faulty_jobs_counter += 1
            # If primary failed, backup remains and is activated. Do not cancel backup.
            remove_finished_or_failed_from_core(c, job)
            # If group is now finally failed, count it as finished.
            if job.shared_tasks.done and job.shared_tasks.failed:
                total_finished_jobs += 1
                if job.finish_time > job.deadline:
                    deadline_missed_counter += 1

    return TSP_violate_counter, total_finished_jobs, deadline_missed_counter, faulty_jobs_counter


In [ ]:

# ============================================================
# Simulation + reports
# ============================================================
def update_all_reliability(data, env):
    for task in data:
        task.update_reliability(env)
    return system_reliability(data)


def write_reports(output_path, jobs, data, cores, power_trace, slack_trace, hyper_period, qos, ENERGY, energy, fault_report, system_r):
    os.makedirs(output_path, exist_ok=True)

    # Detailed job report.
    file_name = os.path.join(output_path, "task_detail.csv")
    with open(file_name, mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow([
            "Name", "Job_id", "copy_type", "core_id", "Arrival", "StartTime", "FinishTime", "Deadline", "Period",
            "v", "f", "useful_work_ms", "checkpointed_WCET_ms", "n_checkpoints", "checkpoint_times_ms",
            "transient_faults", "disruptive_faults", "rollback_events", "reexecuted_work_ms",
            "status", "failure_reason", "peak_p", "avg_p", "min_p"
        ])
        for j in jobs:
            writer.writerow([
                j.name, j.job_id, "primary" if j.is_primary else "backup", j.core.id + 1 if j.core else -1,
                j.arrival, j.start_time, j.finish_time, j.deadline, j.period, j.v, j.f,
                j.useful_work_ms, j.current_exec_time, j.n_checkpoints, j.active_checkpoint_times if j.is_primary else [],
                j.transient_fault_events, j.disruptive_fault_events, j.rollback_events, j.reexecuted_work_ms,
                j.status, j.failure_reason, j.get_peak_p(), j.get_avg_p(), j.get_min_p()
            ])

    # Reliability report.
    file_name = os.path.join(output_path, "reliability_report.csv")
    with open(file_name, mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow([
            "Task", "k_faults", "checkpoint_overhead_ms", "recovery_overhead_ms", "n_checkpoints",
            "checkpoint_times_ms", "WCET_no_cp_ms", "WCET_cp_ms", "R_checkpoint", "R_backup", "R_total"
        ])
        for t in data:
            writer.writerow([
                t.name, t.k_faults, t.checkpoint_overhead, t.recovery_overhead, t.n_checkpoints,
                t.checkpoint_times, t.WCET, t.WCET_cp, t.R_cp, t.R_backup, t.total_reliability
            ])
        writer.writerow([])
        writer.writerow(["System reliability", system_r])
        for k, v in fault_report.items():
            writer.writerow([k, v])

    # Power / energy report.
    file_name = os.path.join(output_path, "power_energy.csv")
    energy_trace = 0
    with open(file_name, mode="w", newline="") as file:
        writer = csv.writer(file)
        header = []
        for i in range(len(cores)):
            header += [f"Core{i+1} load(%)", f"Core{i+1} peak", f"Core{i+1} P", f"Core{i+1} E"]
        writer.writerow(header)
        tr = []
        for c in cores:
            load = round((sum(slack_trace[c.id]) / max(1, len(slack_trace[c.id]))) * 100, 2)
            peak = max(power_trace[c.id]) if power_trace[c.id] else 0
            avg = sum(power_trace[c.id]) / max(1, len(power_trace[c.id]))
            e = avg * hyper_period * 1e-3
            tr += [load, peak, avg, e]
            energy_trace += e
        writer.writerow(tr)

    # Summary.
    file_name = os.path.join(output_path, "information.csv")
    with open(file_name, mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow([
            "ENERGY", "Trace_Energy", "QoS(%)", "System reliability",
            "Faulty jobs", "Transient fault events", "Disruptive fault events",
            "Faulty primary jobs", "Faulty backup jobs", "Successful copy jobs"
        ])
        writer.writerow([
            round(ENERGY, 2), round(energy_trace, 2), round(qos * 100, 3), system_r,
            fault_report["faulty_jobs"], fault_report["transient_fault_events"], fault_report["disruptive_fault_events"],
            fault_report["faulty_primary_jobs"], fault_report["faulty_backup_jobs"], fault_report["successful_jobs"]
        ])


def simulate(env, data, TSP, hyper_period, output_path, log_path, total_jobs):
    cores = [Core(c) for c in range(env.cores)]
    data = sorted(data, key=lambda d: d.WCET_cp / d.period, reverse=True)

    ustate = {i: 0 for i in range(env.cores)}
    map(data, ustate, env, cores)
    print("Core utilizations after primary+backup mapping:", {c.id: round(c.current_utilization, 4) for c in cores})

    system_r = update_all_reliability(data, env)
    jobs, arrivals = create_jobs(data, int(hyper_period), cores, env)

    calculate_slack(cores, hyper_period)
    define_slack(cores, data, hyper_period)

    arrival_queue = deque(arrivals)
    job_queue = deque(sorted(jobs, key=lambda j: (j.arrival, j.deadline, 0 if j.is_primary else 1)))

    power_trace = [[] for _ in cores]
    util_trace = []
    slack_trace = [[] for _ in cores]

    TSP_violate_counter = 0
    total_finished_jobs = 0
    deadline_missed_counter = 0
    faulty_jobs_counter = 0

    fault_report = {
        "faulty_jobs": 0,
        "faulty_primary_jobs": 0,
        "faulty_backup_jobs": 0,
        "transient_fault_events": 0,
        "disruptive_fault_events": 0,
        "successful_jobs": 0,
    }

    max_time = int(hyper_period * 2 + 1)
    for time in range(max_time):
        if time % 100000 == 0:
            print(time, "simulation time")

        if not (total_finished_jobs >= total_jobs and time >= hyper_period):
            for c in cores:
                slack_trace[c.id].append(1 if c.running else 0)

        execute(cores, time, power_trace, util_trace, total_jobs, total_finished_jobs,
                faulty_jobs_counter, hyper_period, env, fault_report)

        rv1, rv2, rv3, rv4 = handle_finished_jobs(cores, time, TSP)
        TSP_violate_counter += rv1
        total_finished_jobs += rv2
        deadline_missed_counter += rv3
        faulty_jobs_counter += rv4

        new_arrived = manage_arrival(arrival_queue, cores, job_queue, time)
        if new_arrived:
            for core in cores:
                core.EDF()

        dispatch(cores, time)
        DVFS(cores, time, env.vf_levels)

        if total_finished_jobs >= total_jobs and time >= hyper_period:
            break

    print("\n=============================")
    print("TSP violations:", TSP_violate_counter)
    print("Finished task instances:", total_finished_jobs)
    print("Deadline misses:", deadline_missed_counter)
    print("Faulty copy jobs:", faulty_jobs_counter)
    print("Fault report:", fault_report)
    print("Analytical system reliability:", system_r)
    print("=============================\n")

    D_main = []
    f_main = []
    for group_job in jobs:
        if group_job.shared_tasks.final_job is group_job and group_job.shared_tasks.success:
            D_main.append(group_job.deadline)
            f_main.append(group_job.finish_time)
    P = [1 for _ in range(len(f_main))]
    qos, UF = qos_system_no_backup(P, f_main, D_main)

    ENERGY = 0
    for j in jobs:
        if j.is_real_exec_set:
            ENERGY += max(0, j.total_runtime_ms) * 1e-3 * j.get_avg_p()

    write_reports(output_path, jobs, data, cores, power_trace, slack_trace, hyper_period,
                  qos, ENERGY, 0, fault_report, system_r)

    return {
        "qos": qos,
        "TSP_violate_counter": TSP_violate_counter,
        "total_finished_jobs": total_finished_jobs,
        "deadline_missed_counter": deadline_missed_counter,
        "faulty_jobs_counter": faulty_jobs_counter,
        "fault_report": fault_report,
        "system_reliability": system_r,
    }


In [ ]:
# Configure and enable the experiment after generating task CSV files.
from pathlib import Path

current = Path.cwd().resolve()
PROJECT_ROOT = current.parent if current.name == "notebooks" else current
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    core_count = 4
    task_count = core_count * 10
    tsp_limit = 10_000
    load_point = 8
    dataset_dir = PROJECT_ROOT / "data" / "generated" / f"tsp-{core_count}c" / f"limit_{(load_point * core_count) / 10}" / "1"
    csv_files = [str(dataset_dir / f"task{index}.csv") for index in range(1, task_count + 1)]

    missing = [path for path in csv_files if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(f"Missing {len(missing)} task files under {dataset_dir}")

    env = Envir(None, None, core_count, len(csv_files), None)
    env.default_k_faults = 2
    env.default_checkpoint_overhead = 1
    env.default_recovery_overhead = 1
    env.transient_fault_ratio = 0.9
    env.disruptive_fault_ratio = 0.1
    env.read_data(csv_files)
    env.reset()
    data = env.train_data.copy()

    hyper_period = lcm_of_list([int(task.deadline) for task in data])
    total_jobs = sum(hyper_period / task.deadline for task in data)
    output_path = PROJECT_ROOT / "artifacts" / "edf-checkpoint" / f"load_{load_point}"
    output_path.mkdir(parents=True, exist_ok=True)

    result = simulate(env, data, tsp_limit, hyper_period, str(output_path) + os.sep, "", total_jobs)
    print(result)
